In [25]:
import pandas as pd
from datetime import datetime, timedelta

In [26]:
### 转换TRACE文件 F:/AAA-ZSY/IT/SYN3-IT-CONS
def process(csv_path):
    df = pd.read_csv(csv_path)
    header_row = df.iloc[0]
    columns_to_remove = [col for col, val in zip(df.columns, header_row) if val.strip().lower() == 'rejected']
    modified_df = df.drop(columns=columns_to_remove)

    return modified_df

In [27]:
df=process('F:/AAA-RXC/PL8-4sessions/TRACE.csv')
df.to_csv('F:/AAA-RXC/PL8-4sessions/TRACE_Con.csv', index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_41208\1896225307.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


In [28]:
# 读取原始CSV文件 "G:\Data-Neuron Processed\PL3\Social\1-PL3-SOCIAL"
csv_file_path = 'F:/AAA-ZSY(2)/SYN6-CON/TRACE.csv'  # 请将'path/to/your/csv_file.csv'替换为您CSV文件的实际路径
df = pd.read_csv(csv_file_path)

# 将'In left', 'In right', 'Investigating left sniff', 'Investigating right sniff'列转换为布尔值
boolean_columns = ['In left', 'In right', 'Investigating LEFT SNIFF', 'Investigating RIGHT SNIFF','In all']
df[boolean_columns] = df[boolean_columns].astype(bool)

# 创建一个新的DataFrame来存储生成的数据
new_df = pd.DataFrame(columns=["From Second", "To Second", "Event"])

# 遍历原始DataFrame的行
event_id = 1  # 初始化事件ID

# 处理第一行，如果一开始标记为1
first_row = df.iloc[0]  # 第一行数据
for column in boolean_columns:
    if first_row[column]:
        event_name = column.replace(" ", "_")  # 使用列名作为事件名称
        from_time = datetime.strptime(first_row["Time"], "%M:%S.%f")  # 解析开始时间字符串   "%H:%M:%S.%f"
        from_second = (from_time - from_time.replace(hour=0, minute=0, second=0, microsecond=0)).total_seconds()  # 转换为浮点数秒

        # 查找事件结束时间
        j = 1
        while j < len(df):
            next_row = df.iloc[j]  # 下一行数据
            if not next_row[column]:
                break
            j += 1
        to_time = datetime.strptime(df.iloc[j - 1]["Time"], "%M:%S.%f")  # 解析结束时间字符串   "%H:%M:%S.%f"
        to_second = (to_time - to_time.replace(hour=0, minute=0, second=0, microsecond=0)).total_seconds()  # 转换为浮点数秒

        # 添加新行到新的DataFrame
        new_row = {"From Second": from_second, "To Second": to_second, "Event": event_name}
        new_df = new_df.append(new_row, ignore_index=True)

        event_id += 1  # 增加事件ID

# 继续遍历原始DataFrame的其余行
for i in range(1, len(df)):
    prev_row = df.iloc[i - 1]  # 前一行数据
    curr_row = df.iloc[i]  # 当前行数据

    # 检查事件是否从0变为1
    for column in boolean_columns:
        if curr_row[column] and not prev_row[column]:
            event_name = column.replace(" ", "_")  # 使用列名作为事件名称
            from_time = datetime.strptime(curr_row["Time"], "%M:%S.%f")  # 解析开始时间字符串   "%H:%M:%S.%f"
            from_second = (from_time - from_time.replace(hour=0, minute=0, second=0, microsecond=0)).total_seconds()  # 转换为浮点数秒

            # 查找事件结束时间
            j = i + 1
            while j < len(df):
                next_row = df.iloc[j]  # 下一行数据
                if not next_row[column]:
                    break
                j += 1
            to_time = datetime.strptime(df.iloc[j - 1]["Time"], "%M:%S.%f")  # 解析结束时间字符串  "%H:%M:%S.%f"
            to_second = (to_time - to_time.replace(hour=0, minute=0, second=0, microsecond=0)).total_seconds()  # 转换为浮点数秒

            # 添加新行到新的DataFrame
            new_row = {"From Second": from_second, "To Second": to_second, "Event": event_name}
            new_df = new_df.append(new_row, ignore_index=True)

            event_id += 1  # 增加事件ID

# 在新的DataFrame中添加ID列
new_df.insert(0, "ID", range(1, event_id))

# 将生成的数据保存为Excel文件（xlsx格式）
output_excel_file_path = 'F:/AAA-ZSY(2)/SYN6-CON/event_Con.xlsx' # 请将'path/to/your/output_file.xlsx'替换为您想要保存新的Excel文件的路径
new_df.to_excel(output_excel_file_path, index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_41208\1752570818.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file_path)


KeyError: "None of [Index(['In left', 'In right', 'Investigating LEFT SNIFF',\n       'Investigating RIGHT SNIFF', 'In all'],\n      dtype='object')] are in the [columns]"